In [ ]:
!pip install transformers datasets rouge-score sacrebleu pandas numpy torch accelerate


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from datasets import Dataset
from rouge_score import rouge_scorer
import sacrebleu

print("✓ Đã import thư viện")


In [ ]:
DATASET_NAME = "train-vit5-dataset" 
PROCESSED_DATA_DIR = f"/kaggle/input/{DATASET_NAME}"
OUTPUT_DIR = "/kaggle/working"

In [ ]:
# Load dữ liệu đã xử lý
train_df = pd.read_csv(f"{PROCESSED_DATA_DIR}/train_dataset_processed.csv")
val_df = pd.read_csv(f"{PROCESSED_DATA_DIR}/val_dataset_processed.csv")
benchmark_df = pd.read_csv(f"{PROCESSED_DATA_DIR}/benchmark_dataset_processed.csv")

print(f"✓ Train: {len(train_df):,} | Val: {len(val_df):,} | Benchmark: {len(benchmark_df):,}")
print(f"  Columns: {train_df.columns.tolist()}")



# Tokenizer


In [ ]:
tokenizer = T5Tokenizer.from_pretrained("VietAI/vit5-base")
print(f"✓ Tokenizer: vocab={tokenizer.vocab_size:,}")


In [ ]:
def preprocess_function(examples):
    inputs = [prefix + doc for (prefix, doc) in zip(examples['task_prefix'], examples['document'])]
    model_inputs = tokenizer(inputs, max_length=192, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples['summary'], max_length=96, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
test_samples = train_df.head(1).to_dict('list')
test_processed = preprocess_function(test_samples)
print(f"✓ Test: input={len(test_processed['input_ids'][0])}, labels={len(test_processed['labels'][0])}")


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
print(f"✓ Dataset: train={len(train_dataset):,}, val={len(val_dataset):,}")

In [ ]:
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
print(f"✓ Train tokenized: {len(train_dataset):,} samples")

In [ ]:
val_dataset = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
print(f"✓ Val tokenized: {len(val_dataset):,} samples")

In [ ]:
# Abstractive Benchmark
abs_benchmark_filtered = benchmark_df[benchmark_df['task_prefix'] == 'abstractive summary: '].copy()
abs_benchmark_dataset = Dataset.from_pandas(abs_benchmark_filtered)
abs_benchmark_dataset = abs_benchmark_dataset.map(preprocess_function, batched=True, remove_columns=abs_benchmark_dataset.column_names)

# Extractive Benchmark
ext_benchmark_filtered = benchmark_df[benchmark_df['task_prefix'] == 'extractive summary: '].copy()
ext_benchmark_dataset = Dataset.from_pandas(ext_benchmark_filtered)
ext_benchmark_dataset = ext_benchmark_dataset.map(preprocess_function, batched=True, remove_columns=ext_benchmark_dataset.column_names)

print(f"✓ Benchmark prepared: Abs={len(abs_benchmark_dataset):,}, Ext={len(ext_benchmark_dataset):,}")

In [ ]:
sample = train_dataset[0]
print(f"✓ Sample: input_ids={len(sample['input_ids'])}, labels={len(sample['labels'])}")

# Trainning


In [ ]:
print("Loading model...")
model = T5ForConditionalGeneration.from_pretrained("VietAI/vit5-base", low_cpu_mem_usage=True)
print(f"✓ Model: {model.num_parameters():,} parameters")

# Clear cache sau khi load model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = rouge.score(label, pred)
        rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
        rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
        rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)
    
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return {
        'rouge1': np.mean(rouge_scores['rouge1']),
        'rouge2': np.mean(rouge_scores['rouge2']),
        'rougeL': np.mean(rouge_scores['rougeL']),
        'bleu': bleu.score / 100.0
    }

In [ ]:
# Cấu hình Training Arguments (cân bằng memory và tốc độ cho GPU T4)
USE_GPU = torch.cuda.is_available()

if USE_GPU:
    # Tăng batch size lên 6-8 để tăng tốc độ tối đa
    train_batch_size, eval_batch_size = 6, 4  # Batch size lớn hơn
    use_fp16, gradient_accumulation_steps, dataloader_workers = True, 2, 0  # Grad acc = 2 (eff_batch=12)
else:
    train_batch_size, eval_batch_size = 1, 1
    use_fp16, gradient_accumulation_steps, dataloader_workers = False, 1, 0

training_args = Seq2SeqTrainingArguments(
    output_dir=f"{OUTPUT_DIR}/vit5-multi-task-summarizer",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=20,  
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    fp16=use_fp16,
    dataloader_num_workers=dataloader_workers,
    report_to="none",
    warmup_steps=500,
    save_steps=500,
    gradient_checkpointing=False, 
    dataloader_pin_memory=False, 
    ddp_find_unused_parameters=False, 
    remove_unused_columns=True, 
)

eff_batch = train_batch_size * gradient_accumulation_steps * (torch.cuda.device_count() if USE_GPU else 1)
print(f"✓ Config: lr={training_args.learning_rate}, epochs={training_args.num_train_epochs}, eff_batch={eff_batch}")


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print(f"✓ Trainer ready: train={len(train_dataset):,}, val={len(val_dataset):,}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Clear GPU cache và set memory allocation strategy
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    import os
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    estimated_time = "~3-5 giờ" 
    print(f"✓ GPU: {torch.cuda.get_device_name(0)} | Est. time: {estimated_time}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"  Free: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB reserved")
else:
    estimated_time = "~10-15 giờ"
    print(f"⚠️  CPU mode | Est. time: {estimated_time}")

eff_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * (torch.cuda.device_count() if torch.cuda.is_available() else 1)
print(f"Training: {len(train_dataset):,} samples | Batch: {training_args.per_device_train_batch_size} | Eval batch: {training_args.per_device_eval_batch_size} | Grad acc: {training_args.gradient_accumulation_steps} | Eff batch: {eff_batch} | Epochs: {training_args.num_train_epochs}")
print(f"⚠️  Kaggle T4 time limit: 9 hours | Estimated: {estimated_time}")
print("Starting training...\n")


In [ ]:
print("\nRunning Baseline Evaluation (Zero-shot)...")
print("Evaluating Abstractive Benchmark...")
abs_baseline = trainer.evaluate(abs_benchmark_dataset)
print(f"✓ Baseline Abs: R1={abs_baseline.get('eval_rouge1', 0):.4f} R2={abs_baseline.get('eval_rouge2', 0):.4f} RL={abs_baseline.get('eval_rougeL', 0):.4f}")

print("Evaluating Extractive Benchmark...")
ext_baseline = trainer.evaluate(ext_benchmark_dataset)
print(f"✓ Baseline Ext: R1={ext_baseline.get('eval_rouge1', 0):.4f} R2={ext_baseline.get('eval_rouge2', 0):.4f} RL={ext_baseline.get('eval_rougeL', 0):.4f}")
print("-" * 60)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n✓ Training complete!")
print(f"  Loss: {train_result.training_loss:.4f}")
print(f"  Time: {train_result.metrics.get('train_runtime', 0)/60:.1f} minutes")


In [ ]:
model_save_path = f"{OUTPUT_DIR}/best_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ Model saved: {model_save_path}")


In [ ]:
# Training history
if hasattr(trainer.state, 'log_history'):
    print("\nTraining History:")
    for log in trainer.state.log_history:
        if 'epoch' in log:
            epoch = log.get('epoch', 'N/A')
            loss = log.get('train_loss', 'N/A')
            r1, r2, rL = log.get('eval_rouge1', 'N/A'), log.get('eval_rouge2', 'N/A'), log.get('eval_rougeL', 'N/A')
            bleu = log.get('eval_bleu', 'N/A')
            def fmt(val):
                return f"{val:.4f}" if isinstance(val, (float, int)) else str(val)
            print(f"  Epoch {fmt(epoch)}: Loss={fmt(loss)} | R1={fmt(r1)} R2={fmt(r2)} RL={fmt(rL)} | BLEU={fmt(bleu)}")


# Benchmark Evaluation


In [ ]:
print(f"Benchmark: {len(benchmark_df):,} samples")

In [ ]:
# Abstractive Benchmark
print(f"Evaluating Abstractive: {len(abs_benchmark_dataset):,} samples...")
abs_results = trainer.evaluate(abs_benchmark_dataset)

print(f"✓ Abstractive: R1={abs_results.get('eval_rouge1', 0):.4f} R2={abs_results.get('eval_rouge2', 0):.4f} RL={abs_results.get('eval_rougeL', 0):.4f} BLEU={abs_results.get('eval_bleu', 0):.4f}")


In [ ]:
# Extractive Benchmark
print(f"Evaluating Extractive: {len(ext_benchmark_dataset):,} samples...")
ext_results = trainer.evaluate(ext_benchmark_dataset)

print(f"✓ Extractive: R1={ext_results.get('eval_rouge1', 0):.4f} R2={ext_results.get('eval_rouge2', 0):.4f} RL={ext_results.get('eval_rougeL', 0):.4f} BLEU={ext_results.get('eval_bleu', 0):.4f}")


In [ ]:
# Summary
results_summary = pd.DataFrame({
    'Task': ['Abstractive', 'Extractive'],
    'ROUGE-1': [abs_results.get('eval_rouge1', 0), ext_results.get('eval_rouge1', 0)],
    'ROUGE-2': [abs_results.get('eval_rouge2', 0), ext_results.get('eval_rouge2', 0)],
    'ROUGE-L': [abs_results.get('eval_rougeL', 0), ext_results.get('eval_rougeL', 0)],
    'BLEU': [abs_results.get('eval_bleu', 0), ext_results.get('eval_bleu', 0)],
    'Samples': [len(abs_benchmark_dataset), len(ext_benchmark_dataset)]
})

print("\n" + "="*60)
print(results_summary.to_string(index=False))
print("="*60)


In [ ]:
MODEL_PATH = f"{OUTPUT_DIR}/best_model"

if os.path.exists(MODEL_PATH):
    print(f"✓ Tìm thấy model tại: {MODEL_PATH}")
    
    loaded_tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
    print(f"✓ Tokenizer đã load: vocab={loaded_tokenizer.vocab_size:,}")
    
    loaded_model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)
    print(f"✓ Model đã load: {loaded_model.num_parameters():,} parameters")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    print(f"✓ Model đã chuyển sang {device}")
else:
    print(f"⚠️  Không tìm thấy model tại: {MODEL_PATH}")


In [ ]:
def summarize_text(
    text: str,
    task_type: str = "abstractive",  # "abstractive" hoặc "extractive"
    model=None,
    tokenizer=None,
    max_length: int = 192,
    max_target_length: int = 96,
    num_beams: int = 4,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    """
    Tóm tắt văn bản sử dụng model đã train.
    
    Args:
        text: Văn bản cần tóm tắt
        task_type: Loại tóm tắt ("abstractive" hoặc "extractive")
        model: Model đã load (nếu None sẽ dùng loaded_model)
        tokenizer: Tokenizer đã load (nếu None sẽ dùng loaded_tokenizer)
        max_length: Độ dài tối đa của input
        max_target_length: Độ dài tối đa của output
        num_beams: Số beams cho beam search
        device: Device để chạy model
    
    Returns:
        Tóm tắt của văn bản
    """
    if model is None:
        model = loaded_model
    if tokenizer is None:
        tokenizer = loaded_tokenizer
    
    prefix = "abstractive summary: " if task_type == "abstractive" else "extractive summary: "
    input_text = prefix + text
    
    inputs = tokenizer(
        input_text,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=2
        )
    
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [ ]:
if os.path.exists(MODEL_PATH):
    sample_text = """
    Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoại giao Phạm Bình Minh đã có cuộc họp với các đối tác quốc tế 
    để thảo luận về hợp tác kinh tế và thương mại. Cuộc họp diễn ra trong bối cảnh Việt Nam đang tích cực 
    mở rộng quan hệ đối ngoại và thu hút đầu tư nước ngoài. Các bên đã đạt được nhiều thỏa thuận quan trọng 
    về thương mại và đầu tư, góp phần thúc đẩy phát triển kinh tế của đất nước.
    """
    
    print("VÍ DỤ 1: ABSTRACTIVE SUMMARIZATION")
    print(f"\nVăn bản gốc:\n{sample_text.strip()}\n")
    
    abstractive_summary = summarize_text(sample_text.strip(), task_type="abstractive")
    print(f"Tóm tắt abstractive:\n{abstractive_summary}\n")
    
    print("VÍ DỤ 2: EXTRACTIVE SUMMARIZATION")
    print(f"\nVăn bản gốc:\n{sample_text.strip()}\n")
    
    extractive_summary = summarize_text(sample_text.strip(), task_type="extractive")
    print(f"Tóm tắt extractive:\n{extractive_summary}\n")
else:
    print("⚠️  Model chưa được lưu. Hãy chạy training và lưu model trước.")
